# Data Validation Report
**Dataset:** `data/processed/flights_weather.csv`  
**Sources:** BTS 2015 Flight Delays + Open-Meteo Historical Weather API  


In [ ]:
import os
from pathlib import Path

import pandas as pd

# Always run from project root regardless of where the notebook is opened from
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

from src.validation.validate import run_all

df = pd.read_csv("data/processed/flights_weather.csv", low_memory=False)
results = run_all(
    df,
    airlines_path=Path("data/external/airlines.csv"),
    airports_path=Path("data/external/airports.csv"),
)
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

## Summary

In [ ]:
summary = pd.DataFrame([
    {'Check': check, 'Passed': '✓' if r['passed'] else '✗', 'Summary': r['summary']}
    for check, r in results.items()
])
summary

## 1. Schema

In [ ]:
d = results['schema']['details']
print(f"Total rows    : {d['total_rows']:,}")
print(f"Total columns : {d['total_columns']}")
print(f"Missing cols  : {d['missing_columns'] or 'None'}")
print(f"Extra cols    : {d['extra_columns'] or 'None'}")
print(f"Dtype issues  : {d['dtype_issues'] or 'None'}")

## 2. Completeness

In [ ]:
results['completeness']['details']['missing_report']

In [ ]:
print('Unexpected nulls (require justification):')
results['completeness']['details']['unexpected_nulls']

## 3. Duplicates

In [ ]:
d = results['duplicates']['details']
print(f"Full-row duplicates    : {d['full_row_duplicates']}")
print(f"Flight-level duplicates: {d['flight_level_duplicates']} (codeshares — expected)")

## 4. Value Ranges

In [ ]:
d = results['ranges']['details']
if d['range_violations']:
    pd.DataFrame(d['range_violations']).T
else:
    print('All values within expected ranges.')
print(f"\nNumeric origin airports (no IATA match) : {d['numeric_origin_airports']:,}")
print(f"Numeric dest airports (no IATA match)   : {d['numeric_dest_airports']:,}")

## 5. Consistency

In [ ]:
d = results['consistency']['details']
if d:
    for k, v in d.items():
        print(f"{k}: {v['count']} rows — {v['description']}")
else:
    print('All consistency checks passed.')

## 6. Outlier Detection (IQR method)

In [ ]:
outlier_rows = [
    {'column': col, **info}
    for col, info in results['outliers']['details'].items()
]
pd.DataFrame(outlier_rows).set_index('column')

## 7. Format Validation

In [ ]:
d = results['formats']['details']
if d:
    for k, v in d.items():
        print(f"{k}: {v['count']} rows — {v['description']}")
else:
    print('All format checks passed.')

## 8. Temporal Validity

In [ ]:
d = results['temporal']['details']
if d['invalid_dates']:
    pd.DataFrame(d['invalid_dates'])
else:
    print('All dates are valid calendar dates.')

## 9. Referential Integrity

In [ ]:
d = results['referential_integrity']['details']
if d:
    for k, v in d.items():
        print(f"{k}: {v['count']} rows — {v['values'][:10]}")
else:
    print('All airline and airport codes match reference files.')

## 10. Cardinality

In [ ]:
rows = [
    {'column': col, 'unique_count': info['unique_count'], 'top_values': list(info['top_5'].keys())}
    for col, info in results['cardinality']['details'].items()
]
pd.DataFrame(rows).set_index('column')

## 11. Statistical Summary

In [ ]:
results['statistics']['details']['stats']

## 12. Target Variable Readiness

In [ ]:
d = results['target']['details']
print(f"Total rows        : {d['total_rows']:,}")
print(f"Cancelled flights : {d['cancelled_count']:,}")
print(f"ARRIVAL_DELAY nulls: {d['arrival_delay_nulls']:,}")
print(f"Unlabellable rows : {d['unlabellable_rows']:,}")
print()

dist = pd.DataFrame([
    {'class': cls, 'count': cnt, 'pct': d['class_pct'][cls]}
    for cls, cnt in d['class_counts'].items()
]).set_index('class')
display(dist)

dist['count'].plot.bar(title='Class Distribution', ylabel='Count', xlabel='Class')